# Chapter 9 — reproducing GPT-2 (124M)

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 9 — reproducing GPT-2 (124M)

**Video:** 4h01m · [youtu.be/l8pRSuU81PU](https://youtu.be/l8pRSuU81PU) · the longest in the series.

**Status note:** the course page lists eight lectures and ends with "ongoing...", so this appears in the YouTube playlist and the GitHub repository but not on the page [uncertain whether that is deliberate or a stale page]. Karpathy opens with "we are going to be continuing our Zero to Hero series" [transcript], so it belongs here.

**A note on verification:** unlike every previous chapter, I did not run this training end to end; it needs 8 GPUs for 1.7 hours. Numbers here are marked [transcript] where they are Karpathy's measurements, and the code is his structure rather than something I executed at full scale. The small pieces marked [verified] I did run.

### The problem

Chapter 7 built a transformer that works. This chapter covers everything between "works" and "competitive": real data, real hyperparameters, real hardware, real evaluation. It ends with a model matching OpenAI's published GPT-2 124M, trained from scratch in about an hour for roughly **$10** of rented cloud GPUs. [transcript]

> **Say it to a six-year-old.** You already built a small toy engine that works. Now you are building the real car: the same engine, but bigger, with proper fuel, and you spend most of your time making sure the fuel gets in fast enough, because the engine keeps sitting there waiting with nothing to burn.

### The target

GPT-2 shipped in four sizes, 124 million up to 1,558 million parameters. [transcript]

| Setting | GPT-2 124M |
|---|---|
| Layers | 12 |
| Embedding dimension | 768 |
| Attention heads | 12 |
| Vocabulary | 50,257 tokens |
| Context length | 1,024 tokens |

**Run it.** Confirm the model's shape yourself, without training anything:

In [ ]:
# pip install transformers
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained("gpt2")     # this IS the 124M model
sd = model.state_dict()
print("total parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")
print("token embedding shape:", tuple(sd['transformer.wte.weight'].shape))
print("embedding parameters: ", sd['transformer.wte.weight'].numel()/1e6, "M")
print("weight tying (input embedding is the output classifier):",
      (sd['transformer.wte.weight'] is sd['lm_head.weight'])
      or sd['transformer.wte.weight'].data_ptr() == sd['lm_head.weight'].data_ptr())

**What you should see:**

**Expected output:**

```
total parameters: 124.439808 M
token embedding shape: (50257, 768)
embedding parameters:  38.597376 M
weight tying (input embedding is the output classifier): True
```

[verified]

**38.6 million of 124.4 million parameters, 31%, are the token embedding table** [transcript], and it is *shared* with the output classifier. That sharing is **weight tying**: both encode "which tokens are similar to which," so using one matrix for both saves 30% of the model and works slightly better.

### Step 1 — match the original exactly, then discard it

Rather than starting from a blank file, Karpathy writes his own GPT class using the same layer names as Hugging Face's released GPT-2, loads OpenAI's actual weights into it, and generates text. If the output matches the reference implementation, the architecture is right. Only then does he throw the weights away and train from scratch.

**This is the most transferable habit in the lecture** [my read]: before optimizing anything, build a check that tells you unambiguously whether you are correct.

The other initialization detail he copies: weights start with standard deviation 0.02, and layers writing into the residual stream are scaled by `1/sqrt(2 × n_layers)`, so that adding many blocks does not let the accumulated signal grow without bound. Note 0.02 is close to `1/sqrt(768) = 0.036`, so it is roughly the Kaiming rule from Chapter 4. [transcript]

### Step 2 — make it fast

Baseline: about 1,000 milliseconds per step. Five changes: [transcript]

| Change | What it does | Result |
|---|---|---|
| **TF32 precision** | Lets the GPU use lower-precision matrix-multiply units | 1,000 ms → ~333 ms |
| **bfloat16** | Lower precision again, for activations | ~333 ms → ~300 ms |
| **`torch.compile`** | Compiles the model into fused kernels, removing Python overhead and keeping intermediate results in fast memory | ~300 ms → ~129 ms |
| **FlashAttention** | Restructured attention that never materializes the T×T matrix in memory | ~130 ms → ~96 ms |
| **"Nice numbers"** | Vocabulary 50,257 → **50,304**, divisible by 128 | ~96 ms → ~93 ms |

Total: about **11× faster**.

**Precision, since it has not been defined.** A **float32** number uses 32 bits and is accurate to about 7 decimal digits. **bfloat16** uses 16, keeping the same range but only 2–3 digits of precision. Neural network training tolerates this because gradients are noisy anyway; halving the bits halves the memory traffic, which is what actually limits speed.

**The vocabulary change deserves attention**, because it is the most surprising item. 50,304 is *more* work in principle, adding 47 tokens the tokenizer can never emit. It runs faster anyway, because GPU kernels are built around power-of-two block sizes and an awkward number forces a slow fallback path for the remainder. Karpathy's heuristic: "scan your code and look for ugly numbers." He notes that on PyTorch 2.3.1 or earlier the same change bought about 30% rather than 4%. [transcript]

**The theme underneath all five:** most of the time the arithmetic units sit idle waiting for data to arrive from memory. The workload is **memory-bound**, not compute-bound, so nearly every optimization moves fewer bytes rather than doing less maths. "If you're getting 60% utilization you're actually doing extremely well." [transcript]

**Run it.** Measure the memory-bound claim on your own GPU:

In [ ]:
import torch, time
if torch.cuda.is_available():
    a = torch.randn(8192, 8192, device='cuda')
    b = torch.randn(8192, 8192, device='cuda')
    for dtype, name in [(torch.float32, 'float32'), (torch.bfloat16, 'bfloat16')]:
        x, y = a.to(dtype), b.to(dtype)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(10):
            _ = x @ y
        torch.cuda.synchronize()
        dt = (time.time() - t0) / 10
        flops = 2 * 8192**3 / dt / 1e12
        print(f"{name}: {dt*1000:.1f} ms per matmul, {flops:.1f} TFLOPS")

**What you should see** (numbers depend entirely on your GPU):

**Expected output:**

```
float32: 157.7 ms per matmul, 7.0 TFLOPS
bfloat16: 34.7 ms per matmul, 31.7 TFLOPS
```

[verified, on the machine this book was written on]

A **4.5× speedup** from changing nothing but the number format, on identical hardware doing identical mathematics. That is the entire argument of step 2 in one measurement. Your absolute numbers will differ; the ratio is the point.

### Step 3 — the training recipe

The GPT-2 paper is vague about training, so Karpathy follows the GPT-3 paper, which is specific: [transcript]

- **AdamW**, betas 0.9 and 0.95, epsilon 1e-8, weight decay 0.1, fused implementation.

**What AdamW is**, since every chapter until now used plain gradient descent: instead of stepping by the gradient alone, Adam keeps two running averages per parameter, one of recent gradients (momentum, which smooths out noise) and one of recent squared gradients (which measures how volatile that parameter has been), and divides by the second. Parameters with consistently small gradients get larger steps; volatile ones get smaller steps. In practice it removes most of the learning-rate sensitivity you saw in Chapter 3's sweep. The **W** is decoupled weight decay: a separate pull of every weight toward zero, which is the Chapter 2 regularization idea. [standard]

- **Gradient clipping at 1.0.** If the total gradient size exceeds 1.0, scale it down. One anomalous batch cannot then wreck the model. (Recall Chapter 5's caveat: this discards information from outliers rather than dampening it.)
- **Cosine learning rate schedule with warmup.** Start near zero, ramp up over the first 375 million tokens (715 steps at this batch size), peak at 6e-4, then decay smoothly to 6e-5, which is 10% of peak.

**Why warm up?** Early gradients point in wildly unreliable directions, and a full-size step taken then can push the model somewhere it takes a long time to escape. **Analogy:** easing out the clutch rather than dropping it.

- **Batch size of 0.5 million tokens**, specifically 2^19 = 524,288. [transcript]

**Gradient accumulation.** A single GPU cannot hold half a million tokens at once. So process a micro-batch of 16 sequences × 1,024 tokens, compute gradients, keep them, repeat 32 times, and only then update. Mathematically identical to one giant batch, serialized in time.

**The detail everyone gets wrong:** you must divide each micro-batch's loss by the number of accumulation steps, because the loss is a *mean* and summing 32 means overcounts by 32×. This is exactly the `/n` from Chapter 5, and getting it wrong silently multiplies your learning rate by 32.

In [ ]:
loss_accum = 0.0
for micro_step in range(grad_accum_steps):
    x, y = train_loader.next_batch()
    logits, loss = model(x, y)
    loss = loss / grad_accum_steps          # <- the line everyone forgets
    loss_accum += loss.detach()
    loss.backward()                          # gradients accumulate across micro-steps
norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
optimizer.zero_grad(set_to_none=True)

**Distributed training.** With 8 GPUs, PyTorch's `DistributedDataParallel` runs a copy of the model on each and averages gradients across them after each backward pass. Throughput reaches roughly **1.5 million tokens per second**. [transcript]

### Step 4 — real data

- **What GPT-2 used:** WebText, built by scraping every outbound Reddit link with at least 3 karma. 45 million links, about 40 GB of text. [transcript]
- **What this lecture uses:** **FineWeb-EDU**, a filtered, education-heavy subset of web crawl data, as a 10-billion-token sample in 100 shards of exactly 100 million tokens each. [transcript]

At 524,288 tokens per step, 10 billion tokens is **19,073 steps**, one pass over the data, about **1.7 hours** on 8 GPUs. [transcript]

The filtering matters more than it looks. Karpathy's read on why his run beats GPT-2 with 10× less data is that FineWeb-EDU is a much cleaner, narrower distribution than raw scraped Reddit links [transcript]. **Data quality substitutes for data quantity**, which is one of the more important practical lessons in the course. [my read]

### Step 5 — evaluate honestly

Validation loss is necessary and not sufficient, so he adds **HellaSwag**, a multiple-choice sentence-completion benchmark that is easy for people and hard for models: [transcript]

| | Score |
|---|---|
| Random guessing | 25% |
| GPT-2 124M | **29.55%** |
| GPT-2 XL (1.5B) | ~49% |
| Humans | 95% |
| State of the art at lecture time | ~95% |

Its virtue here is **smooth early signal**: small models climb 25% → 26% → 27% gradually, so you see progress long before the model is any good. A benchmark that stays flat until the model is strong tells you nothing during training.

### Results

- The 1.7-hour, 10-billion-token run **surpasses OpenAI's GPT-2 124M**, using 10× fewer tokens than GPT-2's roughly 100 billion. [transcript]
- An overnight run, about 8 hours and 4 epochs, roughly 40 billion tokens, approaches **GPT-3 124M**, which was trained on 300 billion. [transcript]

### And a bug he leaves in

The loss curve has strange periodic wobbles. His diagnosis: the 10-billion-token sample "was not properly shuffled" and the data loader marches through documents in fixed order, so each epoch replays the same sequence of topics. He says plainly: "there's some issue here with the data that I don't fully understand yet." [transcript]

Leaving that in is a pedagogical choice worth noticing. Published results rarely show the parts the author has not figured out. [my read]

> **For the PhD in the room.** A few things worth flagging. The token budget here is far off Chinchilla-optimal: Hoffmann et al. (2022) put the compute-optimal ratio near 20 tokens per parameter, so 124M parameters wants ~2.5B tokens, and this run uses 10B, deliberately overtrained because inference cost, not training compute, dominates in practice. The learning-rate schedule is also not faithful to GPT-3: Karpathy notes his decay horizon equals max steps whereas the paper decays to 10% at 260B of a 300B budget [transcript]. On the systems side, the interesting subtlety in gradient accumulation with DDP is that you want `no_sync()` on all but the final micro-step, otherwise you pay an all-reduce per micro-step; and FlashAttention is not an approximation, it is exact attention with a tiled, recomputed softmax that trades FLOPs for HBM traffic, which is why it wins on a memory-bound workload despite doing more arithmetic.

### Exercises

1. **Load GPT-2 and confirm the weight tying** with the code above. Then check whether the same holds for GPT-2 XL.
2. **Run the precision benchmark** on your own hardware and compute your speedup factor.
3. **Find your GPU's "ugly numbers."** Time a matrix multiply at size 50,257 versus 50,304 and see whether you can reproduce the effect.
4. **Implement gradient accumulation** on the Chapter 7 Shakespeare model: micro-batch 16 accumulated 4 times versus batch 64 in one go. Confirm the losses track each other, then omit the `/n` and watch training destabilize.
5. **Compute the Chinchilla-optimal token count** for the Chapter 7 model (10.79M parameters × 20) and compare with what it was actually trained on.

### Troubleshooting

| Symptom | Cause |
|---|---|
| `torch.compile` fails or hangs | Common on older PyTorch or unusual hardware; it is an optimization, so just remove it |
| Loss diverges after adding gradient accumulation | The missing `/grad_accum_steps` |
| Distributed run hangs at startup | All processes must reach every collective operation; an early `return` in one rank deadlocks the rest |
| Throughput far below expectations | Data loading is the bottleneck, not the model; time the loader separately |
| bfloat16 gives `nan` where float32 did not | A genuine overflow; keep the loss and softmax in float32 |

### 30-second version

Take the transformer from Chapter 7, scale it to GPT-2's exact shape, and verify it by loading OpenAI's released weights and checking the output matches before training your own. Five hardware-aware changes make training 11× faster, one of which is padding the vocabulary from 50,257 to 50,304 because GPUs prefer round numbers. After 1.7 hours and about $10 of rented GPUs on 10 billion tokens of filtered educational web text, it beats the original GPT-2 while using a tenth of the training data, because clean data substitutes for lots of data.

---

# PART IV — BEYOND THE COURSE

*Karpathy's numbered course stops where Chapter 9 stops: with a pretrained base model. These two chapters cover the stages that turn that base model into ChatGPT, which he treats in a separate lecture rather than in the nine.*

**What these chapters are grounded in.** Everything before this point came from the nine lectures. These two draw on three sources instead, all cited inline:

- **Karpathy's "Deep Dive into LLMs like ChatGPT"** (3h31m), the lecture where he covers fine-tuning and RL. Marked `[transcript]` as before; the full captions were retrieved and searched the same way.
- **[The Little Book of Reinforcement Learning](https://github.com/alxndrTL/little-book-rl)** by Alexandre Torres Leguet (V1, June 2026), a 154-page introduction that takes RL from the interaction loop through to GRPO and AlphaGo Zero. Claims taken from it are marked `[little-book]` with a section number. It is CC BY-SA 4.0, non-commercial; the prose here is mine, and where I follow its framing I say so.
- **Code I wrote and ran**, marked `[verified]` exactly as before.

**Why these stages are separated from pretraining at all.** Chapter 9's model has read a large slice of the internet and can continue any document plausibly. What it cannot do is be *useful on request*. Those are different skills, learned in different ways, and the difference is the subject of Part IV.

---